In [ ]:
import os
import pandas as pd

# 1. Caminho relativo para a pasta de dados brutos (subindo um nível a partir de 'notebooks')
DATA_RAW_DIR = os.path.join("..", "data", "raw")

# 2. Mapeamento explícito do arquivo para o nome final da variável (Melhor Prática)
# Isso garante controle total sobre o escopo global do seu ambiente de análise
file_mapping = {
    "olist_orders_dataset.csv": "orders",
    "olist_order_items_dataset.csv": "order_items",
    "olist_products_dataset.csv": "products",
    "olist_customers_dataset.csv": "customers",
    "olist_sellers_dataset.csv": "sellers",
    "olist_order_payments_dataset.csv": "payments",
    "olist_order_reviews_dataset.csv": "reviews",
    "olist_geolocation_dataset.csv": "geolocation",
    "product_category_name_translation.csv": "category_translation"
}

print("🔍 Iniciando carga otimizada e desempacotamento dos datasets para EDA...\n")
print(f"📂 Diretório de origem: {os.path.abspath(DATA_RAW_DIR)}\n")

# 3. Loop de leitura e injeção automática no escopo global do Notebook
for filename, var_name in file_mapping.items():
    file_path = os.path.join(DATA_RAW_DIR, filename)
    
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        globals()[var_name] = df
        print(f"✅ Variável '{var_name}' criada com sucesso!")
        print(f"   📊 Formato: {df.shape[0]:,} linhas × {df.shape[1]} colunas\n")
    else:
        print(f"⚠️ Alerta: O arquivo '{filename}' não foi encontrado na pasta 'data/raw/'.\n")

print("Processo concluído! ")

In [ ]:

datasets_to_inspect = [
    "orders", "order_items", "products", "customers", 
    "sellers", "payments", "reviews", "geolocation", "category_translation"
]

print("📊 INICIANDO AUDITORIA DE METADADOS (ESTRUTURA & TIPAGEM) 📊\n")
print("=" * 70)


for var_name in datasets_to_inspect:
    if var_name in globals():
        df = globals()[var_name]
        
        # Cabeçalho do dataset atual
        print(f"📦 DATASET: '{var_name}'")
        print(f"📐 Dimensões: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
        print("-" * 70)
        
        # Criando uma tabela resumo de metadados para este dataset
        metadata_df = pd.DataFrame({
            'Coluna': df.columns,
            'Tipo de Dado': df.dtypes.values,
            'Registros Preenchidos': df.notnull().sum().values,
            'Nulos (%)': (df.isnull().sum().values / len(df) * 100).round(2)
        })
        
        # Exibe o sumário das colunas formatado
        print(metadata_df.to_string(index=False))
        print("\n" + "=" * 70 + "\n")
        
    else:
        print(f"⚠️ Alerta: A variável '{var_name}' não foi encontrada na memória.")
        print("=" * 70 + "\n")

📊 INICIANDO AUDITORIA DE METADADOS (ESTRUTURA & TIPAGEM) 📊

📦 DATASET: 'orders'
📐 Dimensões: 99,441 linhas × 8 colunas
----------------------------------------------------------------------
                       Coluna Tipo de Dado  Registros Preenchidos  Nulos (%)
                     order_id          str                  99441       0.00
                  customer_id          str                  99441       0.00
                 order_status          str                  99441       0.00
     order_purchase_timestamp          str                  99441       0.00
            order_approved_at          str                  99281       0.16
 order_delivered_carrier_date          str                  97658       1.79
order_delivered_customer_date          str                  96476       2.98
order_estimated_delivery_date          str                  99441       0.00


📦 DATASET: 'order_items'
📐 Dimensões: 112,650 linhas × 7 colunas
--------------------------------------------------

In [ ]:


# 1. Converter colunas de data no dataset 'orders'
cols_dates_orders = [
    'order_purchase_timestamp', 'order_approved_at', 
    'order_delivered_carrier_date', 'order_delivered_customer_date', 
    'order_estimated_delivery_date'
]
for col in cols_dates_orders:
    orders[col] = pd.to_datetime(orders[col])

# 2. Converter datas nos outros datasets relevantes
order_items['shipping_limit_date'] = pd.to_datetime(order_items['shipping_limit_date'])
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])

# Vamos cruzar os nulos de entrega com o status do pedido para entender o motivo
null_delivery_status = orders[orders['order_delivered_customer_date'].isnull()]['order_status'].value_counts()

print("Status dos pedidos onde a data de entrega ao cliente está nula:")
print(null_delivery_status)

🛠️ Iniciando a conversão de tipos (Gargalo 1)...
✅ Todas as colunas de data foram convertidas para datetime64!

🕵️ Investigando o Mistério dos Nulos (Gargalo 2)...
Status dos pedidos onde a data de entrega ao cliente está nula:
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64
